## Kanton 2019 — Data Preprocessing

Downloads and preprocesses the scRNA-seq data from Kanton et al. 2019
(*Organoid single-cell genomic atlas uncovers human-specific features of brain development*).

**Source:** ArrayExpress E-MTAB-7552
- `processed.4.zip` → `chimp_cell_counts_consensus.mtx` (~2.1 GB uncompressed)
- `processed.5.zip` → `human_cell_counts_consensus.mtx` (~3 GB uncompressed)
- `processed.6.zip` → `genes_consensus.txt` + gene IDs
- `processed.7.zip` → per-cell metadata (sample, timepoint, cell type)

**Output:** `data/kanton/kanton_preprocessed.h5ad`

**Note on timepoints:** Human and chimp have slightly different day ranges.
We bin cells into broad timepoint groups so both species share the same labels:
- **early** ≈ 30–40 days
- **mid** ≈ 60–75 days  (largest overlap; most cells)
- **late** ≈ 89–131 days

In [ ]:
import sys
from pathlib import Path
import urllib.request
import zipfile
import io
import re

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc

DATA_DIR = Path('../../data/kanton')
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = DATA_DIR / 'kanton_preprocessed.h5ad'

BASE_URL = 'https://ftp.ebi.ac.uk/pub/databases/microarray/data/experiment/MTAB/E-MTAB-7552/'

print('Ready. Output will be saved to:', OUT_PATH)

### Step 1: Download processed data

Downloads the four processed zip files (~2 GB total compressed).
Skips files that are already present on disk.

In [ ]:
def download_file(url, dest):
    dest = Path(dest)
    if dest.exists():
        print(f'  Already exists: {dest.name}')
        return
    print(f'  Downloading {dest.name} ...')
    urllib.request.urlretrieve(url, dest)
    print(f'  Done: {dest.stat().st_size / 1e6:.0f} MB')

files = [
    ('E-MTAB-7552.processed.4.zip', 'chimp_counts.zip'),
    ('E-MTAB-7552.processed.5.zip', 'human_counts.zip'),
    ('E-MTAB-7552.processed.6.zip', 'genes.zip'),
    ('E-MTAB-7552.processed.7.zip', 'metadata.zip'),
]

for remote, local in files:
    download_file(BASE_URL + remote, DATA_DIR / local)

print('All downloads complete.')

### Step 2: Load metadata and extract timepoints

The metadata TSVs contain per-cell info.
We parse the `Barcode` column (which actually stores the sample run ID like `JoC_71d_7k`)
to extract the day number, then bin into broad timepoint groups.

In [ ]:
def parse_metadata(zip_path, fname):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(fname) as f:
            lines = f.read().decode().splitlines()
    # The header has N cols, data has N+1 cols (extra trailing field)
    # Real columns: CellID, Stage, Line(=dev_stage), Sample(=cell_line),
    #               Barcode(=sample_run_id!), actual_barcode, PredCellType, nGene, nUMI, ...
    cols = lines[0].split('\t')
    rows = []
    for line in lines[1:]:
        parts = line.split('\t')
        # parts[4] is the sample_run_id like 'JoC_71d_7k' or 'h409B2_65d_org1'
        cell_id    = parts[0]
        sample_run = parts[4]  # encodes the timepoint
        cell_type  = parts[5] if len(parts) > 5 else ''
        # Extract day number from sample_run
        m = re.search(r'(\d+)d', sample_run)
        day = int(m.group(1)) if m else 0
        rows.append({'cell_id': cell_id, 'sample_run': sample_run,
                     'cell_type': cell_type, 'timepoint_days': day})
    return pd.DataFrame(rows).set_index('cell_id')

meta_chimp = parse_metadata(DATA_DIR / 'metadata.zip', 'metadata_chimp_cells.tsv')
meta_human = parse_metadata(DATA_DIR / 'metadata.zip', 'metadata_human_cells.tsv')

meta_chimp['species'] = 'Chimpanzee'
meta_human['species'] = 'Human'

print('Chimp timepoints:', sorted(meta_chimp['timepoint_days'].unique()))
print('Human timepoints:', sorted(meta_human['timepoint_days'].unique()))
print()

# ── Bin into broad timepoint groups for cross-species alignment ──────────────
# Chimp:  31      61 64 69 71 74  89 120
# Human:  0  32   60 64 65 67    120 128
# Shared bins: 'early'~30-40d, 'mid'~60-75d, 'late'~89-131d
def bin_timepoint(day):
    if 0  <= day <= 40:  return 'early'
    if 41 <= day <= 80:  return 'mid'
    if 81 <= day <= 135: return 'late'
    return 'other'

TP_DAYS = {'early': 35, 'mid': 65, 'late': 110}  # representative days for plotting

meta_chimp['timepoint'] = meta_chimp['timepoint_days'].map(bin_timepoint)
meta_human['timepoint'] = meta_human['timepoint_days'].map(bin_timepoint)

for sp, meta in [('Chimp', meta_chimp), ('Human', meta_human)]:
    print(f'{sp} bins:')
    print(meta.groupby('timepoint').size().to_string())
    print()

### Step 3: Load count matrices

The MTX files are large (~2-3 GB uncompressed).
We use `scipy.sparse` to load them memory-efficiently.

**Warning:** This step may take several minutes and requires ~8 GB RAM.

In [ ]:
import scipy.io

def load_mtx_from_zip(zip_path, mtx_name):
    """Load MTX from inside a zip, via a temporary extracted file."""
    import tempfile, os, shutil
    tmpdir = Path(tempfile.mkdtemp())
    try:
        print(f'  Extracting {mtx_name} ...')
        with zipfile.ZipFile(zip_path) as z:
            z.extract(mtx_name, tmpdir)
        print(f'  Loading matrix ...')
        mat = scipy.io.mmread(tmpdir / mtx_name).tocsr()
        print(f'  Matrix shape: {mat.shape}')
        return mat
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

print('Loading chimp count matrix ...')
chimp_mat = load_mtx_from_zip(DATA_DIR / 'chimp_counts.zip', 'chimp_cell_counts_consensus.mtx')

print('Loading human count matrix ...')
human_mat = load_mtx_from_zip(DATA_DIR / 'human_counts.zip', 'human_cell_counts_consensus.mtx')

# Load gene list
with zipfile.ZipFile(DATA_DIR / 'genes.zip') as z:
    with z.open('genes_consensus.txt') as f:
        gene_lines = f.read().decode().splitlines()

# genes_consensus.txt: EnsemblID \t GeneName
genes_df = pd.DataFrame([l.split('\t') for l in gene_lines],
                        columns=['gene_id', 'gene_name'])
print(f'Genes: {len(genes_df)}')
print(f'Chimp matrix: {chimp_mat.shape} (genes x cells)')
print(f'Human matrix: {human_mat.shape} (genes x cells)')

### Step 4: Build AnnData objects and concatenate

In [ ]:
# MTX files are genes x cells; AnnData wants cells x genes
# Cell IDs are the index of the metadata DataFrames (CC00001, EC00001, ...)

# Verify shapes match metadata
assert chimp_mat.shape[1] == len(meta_chimp), \
    f'Chimp matrix cells {chimp_mat.shape[1]} != metadata rows {len(meta_chimp)}'
assert human_mat.shape[1] == len(meta_human), \
    f'Human matrix cells {human_mat.shape[1]} != metadata rows {len(meta_human)}'
assert chimp_mat.shape[0] == len(genes_df), \
    f'Gene count mismatch: {chimp_mat.shape[0]} != {len(genes_df)}'

import anndata as ad

adata_chimp = ad.AnnData(
    X   = chimp_mat.T,          # cells x genes
    obs = meta_chimp,
    var = genes_df.set_index('gene_id'),
)

adata_human = ad.AnnData(
    X   = human_mat.T,
    obs = meta_human,
    var = genes_df.set_index('gene_id'),
)

# Keep only cells in shared timepoint bins (exclude 'other' and day-0 iPSCs)
adata_chimp = adata_chimp[adata_chimp.obs['timepoint'] != 'other'].copy()
adata_human = adata_human[adata_human.obs['timepoint'] != 'other'].copy()

adata = ad.concat([adata_chimp, adata_human], join='inner')
print(f'Combined: {adata.n_obs:,} cells x {adata.n_vars:,} genes')
print(adata.obs.groupby(['species', 'timepoint']).size().to_string())

### Step 5: Normalise and compute PCA

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=3000, batch_key='species')
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50, use_highly_variable=True)

print('PCA done:', adata.obsm['X_pca'].shape)

adata.write_h5ad(OUT_PATH)
print(f'Saved to {OUT_PATH}  ({OUT_PATH.stat().st_size / 1e6:.0f} MB)')

In [ ]:
print('=== Summary ===')
print(f'Cells: {adata.n_obs:,}')
print(f'Genes: {adata.n_vars:,}')
print(f'PCA dims: {adata.obsm["X_pca"].shape[1]}')
print()
print(adata.obs.groupby(["species", "timepoint"], observed=True).size().to_string())
print()
print('Ready to run chimp_human_parallel.ipynb')